# INT8 dynamic-quantized ONNX accuracy eval on Jigsaw (Colab CPU)

Phase 2, Opt 2 of the `moderation-engine` plan. This notebook quantizes the
ONNX export to INT8 (dynamic, weight-only) and re-runs the same per-label
F1 / precision / recall / ROC-AUC eval used for the PyTorch baseline and
the Opt 1 fp32 ONNX run. Output is compared against `baseline_accuracy.md`
to quantify the accuracy cost of INT8 weights.

Local 200-sample parity check showed max abs prob diff ~0.13, but only
5 / 1200 binary decisions flipped at threshold 0.5 (0.42%) — so most
probability shifts don't cross the moderation boundary.

**Runtime → Change runtime type → CPU** before running cells. ONNX Runtime's
CUDA EP does NOT accelerate dynamically-quantized QInt8 ops
(`MatMulInteger`, `DynamicQuantizeLinear`); they fall back to CPU and the
constant CPU↔GPU tensor copies make a T4 run *slower* than plain CPU. Use
a CPU runtime — Colab's free CPU tier is fine, latency on Colab isn't the
metric anyway (EC2 c6i.large is).

**Outputs** (downloaded back at the end):
- `int8_accuracy.md` — summary table for `docs/`
- `int8_predictions.parquet` — raw per-row probs for parity diffs

In [ ]:
!pip -q install 'transformers>=4.40' 'optimum[onnxruntime]>=1.20' onnxruntime pandas scikit-learn pyarrow tqdm
import onnxruntime as ort
print('onnxruntime', ort.__version__, 'providers:', ort.get_available_providers())

## Upload the Jigsaw test CSVs from your Mac

Run the cell, then in the **Choose Files** dialog select BOTH:

- `~/Desktop/Coding/Moderation Engine/data/jigsaw/test.csv` (~60 MB)
- `~/Desktop/Coding/Moderation Engine/data/jigsaw/test_labels.csv` (~5 MB)

(`⌘`-click to multi-select.)

In [ ]:
import pathlib
from google.colab import files
uploaded = files.upload()
required = {'test.csv', 'test_labels.csv'}
missing = required - set(uploaded)
assert not missing, f'missing files: {missing}'
data = pathlib.Path('data'); data.mkdir(exist_ok=True)
for name in required:
    (data / name).write_bytes(uploaded[name])
!ls -lh data/test.csv data/test_labels.csv

## Export → Quantize

Same fp32 export as Opt 1, then `onnxruntime.quantization.quantize_dynamic`
with `QInt8` weight type (matches `scripts/quantize_onnx.py` locally so
the eval result is reproducible against what the EC2 container runs).

In [ ]:
!python -m optimum.exporters.onnx --model unitary/toxic-bert --task text-classification models/onnx-toxic-bert
import shutil, pathlib
from onnxruntime.quantization import QuantType, quantize_dynamic
src = pathlib.Path('models/onnx-toxic-bert')
out = pathlib.Path('models/onnx-toxic-bert-int8')
out.mkdir(parents=True, exist_ok=True)
quantize_dynamic(model_input=str(src / 'model.onnx'), model_output=str(out / 'model.onnx'), weight_type=QuantType.QInt8)
for name in ('config.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt', 'special_tokens_map.json'):
    p = src / name
    if p.exists():
        shutil.copy2(p, out / name)
!ls -lh models/onnx-toxic-bert/model.onnx models/onnx-toxic-bert-int8/model.onnx

## Run the eval (CPUExecutionProvider on the INT8 graph)

Same logic as the fp32 ONNX run; only the model path changes. Filters to
the 63,978 Kaggle-scored rows, threshold 0.5, batch size 64. Dynamic INT8
ops are CPU-only in ORT, so we pin `CPUExecutionProvider` explicitly.

In [ ]:
import json, os, time, pathlib
import numpy as np
import pandas as pd
import onnxruntime as ort
from transformers import AutoTokenizer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from tqdm.auto import tqdm

ONNX_DIR = pathlib.Path('models/onnx-toxic-bert-int8')
LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
MODEL_NAME = 'unitary/toxic-bert'
BATCH_SIZE = 64

test = pd.read_csv('data/test.csv')
labels = pd.read_csv('data/test_labels.csv')
df = test.merge(labels, on='id', how='inner')
df = df[(df[LABEL_COLS] != -1).all(axis=1)].reset_index(drop=True)
print(f'scored rows: {len(df):,}')

tokenizer = AutoTokenizer.from_pretrained(str(ONNX_DIR))
config = json.loads((ONNX_DIR / 'config.json').read_text())
id2label = config['id2label']
labels_order = [id2label[str(i)] for i in range(len(id2label))]
assert labels_order == LABEL_COLS, labels_order

so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so.intra_op_num_threads = os.cpu_count() or 2
session = ort.InferenceSession(
    str(ONNX_DIR / 'model.onnx'),
    sess_options=so,
    providers=['CPUExecutionProvider'],
)
print('chosen providers:', session.get_providers(), 'threads:', so.intra_op_num_threads)
input_names = {i.name for i in session.get_inputs()}

texts = df['comment_text'].fillna('').tolist()
raw_probs = []
t0 = time.perf_counter()
for s in tqdm(range(0, len(texts), BATCH_SIZE), unit='batch'):
    batch = texts[s:s+BATCH_SIZE]
    enc = tokenizer(batch, return_tensors='np', padding=True, truncation=True, max_length=512)
    feed = {n: enc[n] for n in input_names if n in enc}
    logits = session.run(None, feed)[0]
    raw_probs.extend((1.0 / (1.0 + np.exp(-logits))).tolist())
elapsed = time.perf_counter() - t0
print(f'inference {elapsed:.1f}s ({len(df)/elapsed:.1f} samples/s)')

probs = pd.DataFrame(raw_probs, columns=LABEL_COLS, index=df.index)
preds = (probs >= 0.5).astype(int)
metrics = []
for label in LABEL_COLS:
    metrics.append({
        'label': label,
        'positives': int(df[label].sum()),
        'f1': f1_score(df[label], preds[label], zero_division=0),
        'precision': precision_score(df[label], preds[label], zero_division=0),
        'recall': recall_score(df[label], preds[label], zero_division=0),
        'roc_auc': roc_auc_score(df[label], probs[label]),
    })
macro_f1 = sum(m['f1'] for m in metrics) / len(metrics)
print(f'\nmacro-F1: {macro_f1:.4f}')
for m in metrics:
    print(f"  {m['label']:<14}  F1={m['f1']:.4f}  P={m['precision']:.4f}  R={m['recall']:.4f}  AUC={m['roc_auc']:.4f}")

## Write artifacts and download them back

Move into `docs/` locally so they get committed.

In [ ]:
header = '| Label | Positives | F1 | Precision | Recall | ROC-AUC |\n'
header += '|-------|----------:|---:|----------:|-------:|--------:|\n'
body = '\n'.join(
    f"| `{m['label']}` | {m['positives']:,} | {m['f1']:.4f} | {m['precision']:.4f} | {m['recall']:.4f} | {m['roc_auc']:.4f} |"
    for m in metrics
)
table = header + body
summary = (
    f"# INT8 accuracy ({MODEL_NAME}@int8)\n\n"
    f"- Model: `{MODEL_NAME}` · backend `onnx` · INT8 dynamic quantization (weights only)\n"
    f"- Dataset: Jigsaw Toxic Comment Classification, scored test split ({len(df):,} rows)\n"
    f"- Threshold: 0.5\n"
    f"- Compute: Colab CPU · `onnxruntime` CPUExecutionProvider · batch size {BATCH_SIZE}\n"
    f"- Macro-average F1: **{macro_f1:.4f}**\n"
    f"- Inference time: {elapsed:.1f}s ({len(df)/elapsed:.1f} samples/s)\n\n"
    f"{table}\n"
)
out_table = pathlib.Path('int8_accuracy.md')
out_preds = pathlib.Path('int8_predictions.parquet')
out_table.write_text(summary)
pd.concat([df[['id']].reset_index(drop=True), probs.reset_index(drop=True)], axis=1).to_parquet(out_preds, index=False)
print(f'wrote {out_table} and {out_preds}')

from google.colab import files as colab_files
colab_files.download(str(out_table))
colab_files.download(str(out_preds))

## Local follow-up

Move the two downloaded files into the local repo's `docs/` directory:

```bash
mv ~/Downloads/int8_accuracy.md      docs/int8_accuracy.md
mv ~/Downloads/int8_predictions.parquet docs/int8_predictions.parquet
```

Then back in the local Claude session, ask it to: diff against `docs/baseline_predictions.parquet` for per-label F1 / probability deltas, update `docs/benchmarks.md`, and proceed to the container rebuild + EC2 latency sweep.